# 🏇 Knight's Tour — DFS, BFS, and A* Approaches

## 🎯 Overview

In this notebook, we explore **how to solve the Knight’s Tour problem** using three different search algorithms:

- **Depth-First Search (DFS)**  
- **Breadth-First Search (BFS)**  
- **A\*** (A-star Search)

The goal is to understand how each algorithm approaches the problem and compare their performance in terms of **efficiency, optimality, and exploration strategy**.

---

## ♟️ Problem Definition

The **Knight’s Tour** is a classic problem in pathfinding and graph traversal.  
A knight must move on a chessboard so that it visits cells according to the rules of chess (L-shaped moves).

Traditionally, the **closed Knight’s Tour** requires the knight to:
- Visit every cell **exactly once**, and  
- End in a position from which it could return to the starting cell in one legal knight move.

In this notebook, however, we will focus on an **open variant** of the problem.

---

## 🧩 Expected Algorithm Behavior

Each algorithm will handle this version differently:

- **DFS (Depth-First Search):**  
  Will explore deeply and may not find the shortest path efficiently.  
  It tends to behave like a brute-force search and is not guaranteed to find the minimum-move solution.

- **BFS (Breadth-First Search):**  
  Explores layer by layer, ensuring the **shortest path** is found.  
  However, it can be **computationally expensive** for large boards due to its wide exploration.

- **A\*** (A-star Search):  
  Combines BFS’s completeness with heuristic-driven efficiency.  
  Using an appropriate heuristic (e.g., Euclidean or Manhattan distance to the target), it can find the **least-move solution** more efficiently.

---

## ⚙️ Next Steps

In the following sections, we will:

1. Define the board and knight move logic.  
2. Implement DFS, BFS, and A* step by step.  
3. Visualize and compare their results in terms of:
   - Path length  
   - Number of explored nodes  
   - Execution time  

Let’s begin!


## Game

In [ ]:
import numpy as np
import pandas as pd
from google.colab import output

In [ ]:
class KnightsTour():

  def __init__(self, rows=8, columns=8, board=None, curr_pos=None, visited_cells=None, parent=None, action=None, cost=None):

    self.rows = rows
    self.columns = columns
    self.board = np.zeros((rows, columns), dtype=np.int8) if board is None else board
    self.curr_pos = curr_pos
    self.visited_cells = 0 if visited_cells is None else visited_cells
    # state related vars
    self.parent = parent
    self.action = action
    self.cost = 0 if cost is None else cost

  def create_dof_matrix(self):
    # Get degrees of freedom for each cell
    dof = np.zeros((self.rows, self.columns), dtype=np.int8)
    for i in range(self.rows):
      for j in range(self.columns):
        dof[i, j] = len(self.get_knight_moves((i, j)))
    return dof

  @property
  def config(self):
    # Method 1.
    #mask = "".join(map(str, self.board.flatten()))
    #if self.curr_pos is None: return mask
    #return mask+str(self.curr_pos)
    # Method 2
    mask = self.board.tobytes()
    if self.curr_pos is None: return mask
    return (mask, *self.curr_pos)

  def get_knight_moves(self, pos=None):
    if pos is None: pos = self.curr_pos
    if pos is None: return []
    x, y = pos
    moves = []
    # Define possible knight moves
    knight_moves = [(2, 1), (2, -1), (-2, 1), (-2, -1), (1, 2), (1, -2), (-1, 2), (-1, -2)]

    for dx, dy in knight_moves:
      new_x, new_y = x + dx, y + dy

      # Check if the new position is within the board boundaries and not visited
      if 0 <= new_x < self.rows and 0 <= new_y < self.columns and self.board[new_x, new_y] == 0:
        moves.append((new_x, new_y))

    return moves

  def get_available_moves(self):
    # If board is empty can play anywhere
    if self.visited_cells == 0:
      return [(i, j) for i in range(self.rows) for j in range(self.columns)]
    # Otherwise calculare L shaped knight moves from curr_pos
    return self.get_knight_moves()

  def is_done(self):
    return all(self.board.flatten() != 0)

  def get_player_input(self):
    while True:
      try:
        output.clear()
        print(self.board)
        print(f"Available moves: {self.get_available_moves()}")
        pos = input("Select a pos to move the knight to in format x,y:")
        x, y = map(int, pos.split(','))
        assert (x, y) in self.get_available_moves()
        break
      except AssertionError:
        print("Invalid move")
        continue
      except ValueError:
        print("Not a coord")
        continue
    return x, y

  def play(self):
    while True:
      x, y = self.get_player_input()
      self.visited_cells += 1
      self.board[x, y] = self.visited_cells
      self.curr_pos = (x, y)
      # Check win
      if self.is_done():
        print("You won!")
        break
      # Check loose
      if len(self.get_available_moves()) == 0:
        print("You lost!")
        break

  # Util for search algos
  def expand(self):
    children = []
    for move in self.get_available_moves():
      child = KnightsTour(self.rows, self.columns, self.board.copy(), move, self.visited_cells + 1, self, move, self.cost + 1)
      child.board[move] = child.visited_cells
      children.append(child)
    return children

  def __lt__(self, other):
    return False

In [ ]:
# You can test the game manually
game = KnightsTour(rows=3, columns=4)
#game.play()

##Utils

In [ ]:
import heapq

class QueueFrontier():

  def __init__(self):
    self.data = []
    self.configs = set()

  def push(self, value):
    self.data.append(value)
    self.configs.add(value.config)

  def pop(self):
    value = self.data.pop(0)
    self.configs.remove(value.config)
    return value

  def empty(self):
    return len(self.data) == 0

  def __len__(self):
    return len(self.data)

  def __contains__(self, value):
    return value.config in self.configs

class StackFrontier(QueueFrontier):

  def pop(self):
    value = self.data.pop()
    self.configs.remove(value.config)
    return value

class PriorityQueueFrontier(QueueFrontier):

  def push(self, value):
    heapq.heappush(self.data, value)
    self.configs.add(value[1].config)

  def pop(self):
    while self.data:
      priority, state = heapq.heappop(self.data)  # always get the smallest priority
      if state.config in self.configs:           # check if still valid
        self.configs.remove(state.config)      # mark as removed
        return (priority, state)
    raise Exception("Empty queue")

  def decrease_key(self, value):
    heapq.heappush(self.data, value)
    self.configs.add(value[1].config)

## Algorithms

In [ ]:
import time

class Solver():

  def __init__(self, initial_state, max_time=100):
    self.initial_state = initial_state
    self.max_time = max_time
    self.reset_stats()

  def check_time(self):
    self.elapsed = time.time() - self.start_time
    if self.elapsed > self.max_time:
      self.search_depth = self.cost_of_path = None
      raise TimeoutError("Time limit exceeded")

  def reset_stats(self):
    self.search_depth = 0
    self.cost_of_path = 0
    self.max_search_depth = 0
    self.nodes_expanded = 0
    self.elapsed = 0

  @staticmethod
  def backtrack(actual_state):

    path = [actual_state]
    while actual_state.parent != None:
        path.append(actual_state.parent)
        actual_state = actual_state.parent

    path.reverse()
    actions = [state.action for state in path]

    return actions[1:]

  @staticmethod
  def calculate_total_cost(state):
    heuristic = -len(state.get_knight_moves())
    return heuristic

  def solve(self, method="bfs"):
    self.start_time = time.time()
    result = None
    if method == "bfs":
      result = self.__bfs_search()
    elif method == "dfs":
      result = self.__dfs_search()
    elif method == "ast":
      result = self.__a_star_search()
    else:
      raise Exception("Invalid method")

    self.elapsed = time.time() - self.start_time
    return result

  def __bfs_search(self):

    self.reset_stats()

    frontier = QueueFrontier()
    frontier.push(self.initial_state)
    explored = set()

    while not frontier.empty():
      state = frontier.pop()
      explored.add(state.config)

      if state.is_done():
        self.cost_of_path = self.search_depth = state.cost
        path_to_goal = Solver.backtrack(state)
        return path_to_goal

      for neighbor in state.expand():
        if (neighbor not in frontier) and (neighbor.config not in explored):
          frontier.push(neighbor)
          self.max_search_depth = max(self.max_search_depth, neighbor.cost)

      self.nodes_expanded += 1
      self.check_time()

    raise Exception("No solution found")

  def __dfs_search(self):

    self.reset_stats()

    frontier = StackFrontier()
    frontier.push(self.initial_state)
    explored = set()

    while not frontier.empty():
      state = frontier.pop()
      explored.add(state.config)

      if state.is_done():
        self.cost_of_path = self.search_depth = state.cost
        path_to_goal = Solver.backtrack(state)
        return path_to_goal

      for neighbor in reversed(state.expand()):
        if (neighbor not in frontier) and (neighbor.config not in explored):
          frontier.push(neighbor)
          self.max_search_depth = max(self.max_search_depth, neighbor.cost)

      self.nodes_expanded += 1
      self.check_time()

    raise Exception("No solution found")

  def __a_star_search(self):

    self.reset_stats()

    frontier = PriorityQueueFrontier()
    frontier.push((self.calculate_total_cost(self.initial_state), self.initial_state))
    explored = set()

    while not frontier.empty():
      priority, state = frontier.pop()
      explored.add(state.config)

      if state.is_done():
        self.cost_of_path = self.search_depth = state.cost
        path_to_goal = Solver.backtrack(state)
        return path_to_goal

      for neighbor in state.expand():
        if (neighbor not in frontier) and (neighbor.config not in explored):
          heuristic_cost = self.calculate_total_cost(neighbor)
          frontier.push((heuristic_cost + neighbor.cost, neighbor))
          self.max_search_depth = max(self.max_search_depth, neighbor.cost)
        elif neighbor in frontier:
          heuristic_cost = self.calculate_total_cost(neighbor)
          frontier.decrease_key((heuristic_cost + neighbor.cost, neighbor))

      self.nodes_expanded += 1
      self.check_time()

    raise Exception("No solution found")

In [ ]:
# Testing algos

game = KnightsTour(rows=3, columns=4)
solver = Solver(game)

try:
  path = solver.solve(method="ast")
except TimeoutError:
  path = None

print("Solver Solution", "-"*10)
print("* Path:", path)
print("* Search depth:", solver.search_depth)
print("* Cost of path:", solver.cost_of_path)
print("* Max search depth:", solver.max_search_depth)
print("* Nodes expanded:", solver.nodes_expanded)
print("* Elapsed time:", f"{solver.elapsed:.3f}")

Solver Solution ----------
* Path: [(0, 0), (1, 2), (2, 0), (0, 1), (1, 3), (2, 1), (0, 2), (1, 0), (2, 2), (0, 3), (1, 1), (2, 3)]
* Search depth: 12
* Cost of path: 12
* Max search depth: 12
* Nodes expanded: 745
* Elapsed time: 0.019


## Benchmark

In [ ]:
sizes_to_test = [
    (3, 4), (3, 7), (4, 5), (5, 5), (5, 6), (6, 6)
]

metrics = []
for size in sizes_to_test:
  rows, columns = size
  game = KnightsTour(rows=rows, columns=columns)
  solver = Solver(game)
  for method in ["bfs", "dfs", "ast"]:
    print(f"Solving {size} with {method}")
    try:
      path = solver.solve(method=method)
    except TimeoutError:
      path = None
    metrics.append({
        "Board Size": size,
        "Algorithm": method,
        "Search depth": solver.search_depth,
        "Cost of path": solver.cost_of_path,
        "Max search depth": solver.max_search_depth,
        "Nodes expanded": solver.nodes_expanded,
        "Elapsed time": solver.elapsed,
        "Solved": path is not None
    })

report = pd.DataFrame(metrics)

Solving (3, 4) with bfs
Solving (3, 4) with dfs
Solving (3, 4) with ast
Solving (3, 7) with bfs
Solving (3, 7) with dfs
Solving (3, 7) with ast
Solving (4, 5) with bfs
Solving (4, 5) with dfs
Solving (4, 5) with ast
Solving (5, 5) with bfs
Solving (5, 5) with dfs
Solving (5, 5) with ast
Solving (5, 6) with bfs
Solving (5, 6) with dfs
Solving (5, 6) with ast
Solving (6, 6) with bfs
Solving (6, 6) with dfs
Solving (6, 6) with ast


In [ ]:
import matplotlib.colors as mcolors

# Function to color based on solved
def solved_color(val):
  if val:
    return 'background-color: lightgreen'
  else:
    return 'background-color: lightcoral'

# Function to highlight best per board size (lowest Nodes expanded)
def best_per_board(df):
  result = pd.DataFrame('', index=df.index, columns=df.columns)
  for board in df["Board Size"].unique():
    mask = df["Board Size"] == board
    idx_min = df.loc[mask, "Elapsed time"].idxmin()
    if df.loc[idx_min, "Solved"]:
      result.loc[idx_min, :] = 'background-color: limegreen; font-weight: bold'
  return result

light_cmap = mcolors.LinearSegmentedColormap.from_list(
    'light_red_green', ['#ccffcc', '#ffcccc']
)

# Apply styles
styled = (
    report
    .style.applymap(solved_color, subset=['Solved'])
    .background_gradient(subset=['Elapsed time'], cmap=light_cmap)
    .apply(best_per_board, axis=None)
)

styled

/tmp/ipython-input-3699450080.py:27: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .style.applymap(solved_color, subset=['Solved'])


,Board Size,Algorithm,Search depth,Cost of path,Max search depth,Nodes expanded,Elapsed time,Solved
0,"(3, 4)",bfs,12.000000,12.000000,12,745,0.027313,True
1,"(3, 4)",dfs,12.000000,12.000000,12,62,0.002655,True
2,"(3, 4)",ast,12.000000,12.000000,12,745,0.056468,True
3,"(3, 7)",bfs,21.000000,21.000000,21,243594,5.489600,True
4,"(3, 7)",dfs,21.000000,21.000000,21,3835,0.073351,True
5,"(3, 7)",ast,21.000000,21.000000,21,243594,6.788567,True
6,"(4, 5)",bfs,20.000000,20.000000,20,605545,22.747967,True
7,"(4, 5)",dfs,20.000000,20.000000,20,3209,0.042480,True
8,"(4, 5)",ast,20.000000,20.000000,20,605545,17.818192,True
9,"(5, 5)",bfs,nan,nan,12,552300,100.000158,False


**Knight’s Tour algorithm benchmark key insights**



*   For small boards, DFS explores far fewer nodes than BFS or A*, making it the fastest in practice, even though BFS and A* expand similar numbers of nodes. A* overhead does not pay off on small boards.
*   As board size grows, DFS can still succeed on some medium boards, but BFS and A* fail due to memory or time constraints. BFS suffers heavily from node explosion; A* may suffer from heuristic overhead or poor pruning.
*   DFS is the only algorithm able to solve larger boards reliably, despite exploring many nodes. BFS and A* are limited by either memory or runtime overhead.
*  Node explosion is the main bottleneck for BFS and A* on larger boards; careful state representation (e.g., bitmask) and pruning are key to scalability.